# Proyecto Final: Sistema de Identificación del Género Musical

**Autoras:** Laura Monzó, Carla Ortega, Alicia Carmona, Héctor Ruíz  
**Asignatura:** Proyecto I: Introducción a la IA  
**Dataset:** GTZAN Genre Collection (Data-2)  

---

Este notebook integra el trabajo completo del proyecto:
- **Parte 1** — Análisis Exploratorio de Datos (6 fases) sobre `features_30_sec.csv`
- **Parte 2** — Preprocesado de audio y extracción de MFCC desde `genres_original/`
- **Parte 3** — Entrenamiento y análisis de la red neuronal
- **Parte 4** — Análisis avanzado: SHAP, umbrales de confianza
- **Parte 5** — Sistema de predicción y exportación del modelo

In [ ]:
# ─── INSTALACIÓN DE DEPENDENCIAS ────────────────────────────────────────────
import subprocess, sys
pkgs = ['librosa', 'tensorflow', 'seaborn', 'scikit-learn', 'shap', 'scipy']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=False)
print('Dependencias listas.')

In [ ]:
# ─── IMPORTACIONES ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import glob
import os

import librosa
import librosa.display

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

# Rutas al dataset Data-2
DATA_CSV    = 'Data-2/features_30_sec.csv'
DATA_AUDIO  = 'Data-2/genres_original/'
DATA_IMAGES = 'Data-2/images_original/'

GENRES = ['blues','classical','country','disco','hiphop',
          'jazz','metal','pop','reggae','rock']

print('Versión TensorFlow:', tf.__version__)
print('Importaciones correctas.')

---
# PARTE 1: ANÁLISIS EXPLORATORIO DE DATOS (EDA)

Aplicamos las 6 fases del EDA sobre el archivo `features_30_sec.csv`, que contiene 58 características acústicas ya extraídas (MFCC, chroma, spectral centroid, tempo…) para cada una de las ~1.000 canciones del dataset.

## Fase 1: Accesibilidad y Preparación

In [ ]:
df = pd.read_csv(DATA_CSV)

# Limpieza básica
df.columns = df.columns.str.strip()
df = df.dropna(axis=1, how='all')

print('Dimensiones del dataset:', df.shape)
print('\nPrimeras filas:')
display(df.head())
print('\nInformación general:')
display(df.info())
print('\nDistribución de géneros:')
display(df['label'].value_counts())

## Fase 2: Análisis Univariante

In [ ]:
# Estadísticos descriptivos de las variables numéricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
display(df[num_cols].describe().round(4))

# Distribución del género (clase objetivo)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=sns.color_palette('tab10', 10))
axes[0].set_title('Distribución de géneros musicales')
axes[0].set_xlabel('Género')
axes[0].set_ylabel('Número de canciones')
axes[0].tick_params(axis='x', rotation=45)

# Histograma del tempo
axes[1].hist(df['tempo'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución del Tempo (BPM)')
axes[1].set_xlabel('Tempo (BPM)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('eda_univariante_genero_tempo.png', dpi=150)
plt.show()

In [ ]:
# Distribución de MFCC1 y Spectral Centroid por género
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='label', y='mfcc1_mean', ax=axes[0],
            hue='label', palette='tab10', legend=False)
axes[0].set_title('MFCC1 (media) por género')
axes[0].set_xlabel('Género')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df, x='label', y='spectral_centroid_mean', ax=axes[1],
            hue='label', palette='tab10', legend=False)
axes[1].set_title('Spectral Centroid (media) por género')
axes[1].set_xlabel('Género')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_boxplots_features.png', dpi=150)
plt.show()

## Fase 3: Análisis Multivariante

In [ ]:
# Matriz de correlación (features numéricas clave)
key_features = ['chroma_stft_mean','rms_mean','spectral_centroid_mean',
                'spectral_bandwidth_mean','rolloff_mean','zero_crossing_rate_mean',
                'tempo','mfcc1_mean','mfcc2_mean','mfcc3_mean']

corr = df[key_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Correlación — Features acústicas principales')
plt.tight_layout()
plt.savefig('eda_correlacion.png', dpi=150)
plt.show()

In [ ]:
# Dispersión: Spectral Centroid vs ZCR coloreada por género
plt.figure(figsize=(10, 6))
palette = sns.color_palette('tab10', 10)
for i, genre in enumerate(GENRES):
    mask = df['label'] == genre
    plt.scatter(df.loc[mask, 'spectral_centroid_mean'],
                df.loc[mask, 'zero_crossing_rate_mean'],
                label=genre, alpha=0.6, color=palette[i], s=30)
plt.xlabel('Spectral Centroid (media)')
plt.ylabel('Zero Crossing Rate (media)')
plt.title('Separación de géneros en el espacio de features')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('eda_scatter_features.png', dpi=150)
plt.show()

In [ ]:
# Tabulación cruzada: tempo medio por género
tempo_stats = df.groupby('label')['tempo'].agg(['mean','std','min','max']).round(2)
print('Estadísticos de Tempo por género:')
display(tempo_stats)

## Fase 4: Evaluación de Supuestos (Normalidad)

In [ ]:
# Test de Shapiro-Wilk para las features principales
print('Test de Shapiro-Wilk (H0: distribución normal):')
print(f'{"Feature":<35} {"Estadístico":>12} {"p-valor":>12} {"Normal?":>10}')
print('-' * 72)

for col in ['tempo', 'mfcc1_mean', 'spectral_centroid_mean',
            'rms_mean', 'chroma_stft_mean']:
    data_clean = df[col].dropna()
    stat, p = stats.shapiro(data_clean[:5000])  # Shapiro limit 5000
    normal = 'Sí' if p > 0.05 else 'No'
    print(f'{col:<35} {stat:>12.4f} {p:>12.4e} {normal:>10}')

# Q-Q plot para el tempo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stats.probplot(df['tempo'].dropna(), dist='norm', plot=axes[0])
axes[0].set_title('Q-Q Plot — Tempo')
stats.probplot(df['mfcc1_mean'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot — MFCC1 (media)')
plt.tight_layout()
plt.savefig('eda_qqplots.png', dpi=150)
plt.show()

## Fase 5: Detección de Atípicos

In [ ]:
# Método IQR para detectar outliers en features clave
def detectar_outliers_iqr(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return ((series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)).sum()

print('Número de atípicos (IQR) por feature:')
for col in key_features:
    n = detectar_outliers_iqr(df[col])
    pct = n / len(df) * 100
    print(f'  {col:<35}: {n:>3} ({pct:.1f}%)')

# Visualización general de dispersión
features_plot = ['tempo','rms_mean','spectral_centroid_mean',
                 'zero_crossing_rate_mean','mfcc1_mean']
fig, axes = plt.subplots(1, len(features_plot), figsize=(16, 4))
for ax, col in zip(axes, features_plot):
    ax.boxplot(df[col].dropna())
    ax.set_title(col.replace('_mean','').replace('_', ' '), fontsize=9)
plt.suptitle('Detección de Atípicos por Rango Intercuartílico (IQR)')
plt.tight_layout()
plt.savefig('eda_outliers.png', dpi=150)
plt.show()

## Fase 6: Tratamiento de Valores Ausentes

In [ ]:
# Análisis de valores nulos
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

print('Columnas con valores nulos:')
if missing.sum() == 0:
    print('  No hay valores nulos en el dataset. El CSV ya viene limpio.')
else:
    display(pd.DataFrame({'Nulos': missing[missing > 0], '% del total': missing_pct[missing > 0]}))

# Mapa de calor de nulos
plt.figure(figsize=(12, 3))
sns.heatmap(df.isna(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Mapa de Calor de Datos Ausentes (amarillo = ausente)')
plt.tight_layout()
plt.savefig('eda_missing_values.png', dpi=150)
plt.show()

print('\nEstrategia: El dataset GTZAN no presenta valores ausentes.')
print('En caso de encontrarlos: imputar numéricas con mediana, categóricas con moda.')

---
# PARTE 2: PREPROCESADO DE AUDIO — EXTRACCIÓN DE MFCC

Extraemos los coeficientes MFCC directamente de los archivos de audio en `Data-2/genres_original/`.

In [ ]:
def display_mfcc(song_path, title='MFCC'):
    """Carga un .wav y visualiza su espectrograma MFCC."""
    y, sr = librosa.load(song_path)
    mfcc = librosa.feature.mfcc(y=y, sr=sr)
    
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mfcc, x_axis='time', y_axis='mel', sr=sr)
    plt.colorbar(format='%+2.0f')
    plt.title(title)
    plt.tight_layout()
    plt.show()
    print(f'Forma MFCC: {mfcc.shape}  |  Ruta: {song_path}')
    return mfcc

In [ ]:
# Visualización de dos géneros contrastados
blues_sample = glob.glob(DATA_AUDIO + 'blues/*.wav')[0]
classical_sample = glob.glob(DATA_AUDIO + 'classical/*.wav')[0]

display_mfcc(blues_sample, 'MFCC — Blues (percusión, bajas frecuencias)')
display_mfcc(classical_sample, 'MFCC — Classical (timbres suaves, frecuencias medias-altas)')

In [ ]:
# También podemos visualizar el espectrograma mel de las imágenes del dataset
from PIL import Image

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, genre in enumerate(GENRES):
    img_path = glob.glob(DATA_IMAGES + genre + '/*.png')[0]
    img = Image.open(img_path)
    axes[i//5][i%5].imshow(img)
    axes[i//5][i%5].set_title(genre, fontsize=10)
    axes[i//5][i%5].axis('off')
plt.suptitle('Espectrogramas Mel — Un ejemplo por género', fontsize=14)
plt.tight_layout()
plt.savefig('spectrograms_example.png', dpi=150)
plt.show()

In [ ]:
def extract_features_song(f, max_len=25000):
    """Extrae MFCC de un .wav, normaliza a [-1,1] y recorta/rellena a max_len."""
    y, _ = librosa.load(f, mono=True)
    mfcc = librosa.feature.mfcc(y=y)
    mfcc = mfcc.flatten()
    if len(mfcc) < max_len:
        mfcc = np.pad(mfcc, (0, max_len - len(mfcc)), 'constant')
    else:
        mfcc = mfcc[:max_len]
    max_val = np.max(np.abs(mfcc))
    if max_val > 0:
        mfcc = mfcc / max_val
    return mfcc


def generate_features_and_labels(data_path=DATA_AUDIO, genres=GENRES):
    """Procesa todas las canciones, extrae MFCC y genera etiquetas one-hot."""
    all_features, all_labels = [], []
    
    for genre in genres:
        files = glob.glob(os.path.join(data_path, genre, '*.wav'))
        print(f'  {genre:12s}: {len(files)} canciones')
        for f in files:
            try:
                all_features.append(extract_features_song(f))
                all_labels.append(genre)
            except Exception as e:
                print(f'    SKIP {f}: {e}')
    
    label_uniq_ids, label_row_ids = np.unique(all_labels, return_inverse=True)
    labels_oh = to_categorical(label_row_ids, len(label_uniq_ids))
    features = np.stack(all_features)
    
    print(f'\nFeatures: {features.shape}  |  Labels: {labels_oh.shape}')
    return features, labels_oh, label_uniq_ids


print('Extrayendo MFCC de Data-2/genres_original/ ...')
features, labels, genre_names = generate_features_and_labels()

In [ ]:
# División train/test (80/20) con shuffle
alldata = np.column_stack((features, labels))
alldata = shuffle(alldata, random_state=42)

split = int(len(alldata) * 0.8)
train_data, test_data = alldata[:split], alldata[split:]

n_classes = len(genre_names)
train_X, train_y = train_data[:, :-n_classes], train_data[:, -n_classes:]
test_X,  test_y  = test_data[:, :-n_classes],  test_data[:, -n_classes:]

print(f'Train: {train_X.shape[0]} muestras  |  Test: {test_X.shape[0]} muestras')
print(f'Dimensión de entrada: {train_X.shape[1]}')

---
# PARTE 3: ENTRENAMIENTO DE LA RED NEURONAL

In [ ]:
# Arquitectura del modelo
model = Sequential([
    Dense(100, activation='relu', input_shape=(train_X.shape[1],)),
    Dense(10,  activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Entrenamiento
history = model.fit(
    train_X, train_y,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluación en test
loss, acc = model.evaluate(test_X, test_y, verbose=0)
print(f'Pérdida en test:   {loss:.4f}')
print(f'Precisión en test: {acc:.4f} ({acc*100:.1f}%)')

---
# PARTE 4: ANÁLISIS AVANZADO

In [ ]:
# ── Curvas de aprendizaje ──────────────────────────────────────────────────
import matplotlib.ticker as ticker

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes,
                              ['loss', 'accuracy'],
                              ['Pérdida (Loss)', 'Precisión (Accuracy)']):
    ax.plot(history.history[metric],         label='Entrenamiento', marker='o')
    ax.plot(history.history['val_'+metric],  label='Validación',    marker='s')
    ax.set_title(f'¿Cómo aprende la IA? — {title}')
    ax.set_xlabel('Época')
    ax.set_ylabel(title)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_aprendizaje.png', dpi=150)
plt.show()

In [ ]:
# ── Matriz de confusión ────────────────────────────────────────────────────
y_pred  = model.predict(test_X, verbose=0)
pred_cls = np.argmax(y_pred, axis=1)
true_cls = np.argmax(test_y,  axis=1)

cm = confusion_matrix(true_cls, pred_cls)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=genre_names, yticklabels=genre_names)
plt.title('Mapa de Aciertos y Errores (Matriz de Confusión)')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=150)
plt.show()

print('\nInforme de clasificación:')
print(classification_report(true_cls, pred_cls, target_names=genre_names))

In [ ]:
# ── Umbrales de confianza ─────────────────────────────────────────────────
probs = model.predict(test_X, verbose=0)
conf  = np.max(probs, axis=1)
preds_all = np.argmax(probs, axis=1)
correctos = (preds_all == true_cls)

umbrales = [0.99, 0.95, 0.90, 0.80, 0.70, 0.50]
res = []
n_total = len(test_X)

for u in umbrales:
    pasan = conf >= u
    n = np.sum(pasan)
    aciertos = np.sum(correctos[pasan]) if n > 0 else 0
    fallos = n - aciertos
    descartes = n_total - n
    precision = (aciertos / n * 100) if n > 0 else 100
    res.append([u*100, aciertos, fallos, descartes, precision])

df_res = pd.DataFrame(res, columns=['Umbral %','Aciertos','Errores','Para revisión humana','Precisión %'])
print('Análisis de confianza por umbral:')
display(df_res)

fig, ax1 = plt.subplots(figsize=(10, 5))
x = range(len(df_res))
ax1.bar(x, df_res['Aciertos'],  color='#2ecc71', label='Aciertos')
ax1.bar(x, df_res['Errores'],   bottom=df_res['Aciertos'], color='#e74c3c', label='Errores')
ax1.bar(x, df_res['Para revisión humana'],
        bottom=df_res['Aciertos']+df_res['Errores'], color='#dfe6e9', label='Revisión humana')
ax2 = ax1.twinx()
ax2.plot(x, df_res['Precisión %'], color='#2980b9', marker='o', label='Calidad IA (%)')
ax1.set_xticks(x)
ax1.set_xticklabels([f"{int(u)}%" for u in df_res['Umbral %']])
ax1.set_title('Seguridad: Precisión vs Automatización')
ax1.set_xlabel('Umbral de confianza')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('analisis_confianza.png', dpi=150)
plt.show()

In [ ]:
# ── Análisis SHAP ──────────────────────────────────────────────────────────
import shap

# Usamos un subconjunto pequeño para SHAP (es computacionalmente costoso)
X_shap_bg   = train_X[:100]   # fondo para el explainer
X_shap_test = test_X[:5]      # muestras a explicar

explainer   = shap.DeepExplainer(model, X_shap_bg)
shap_values = explainer.shap_values(X_shap_test)

# Importancia media de cada feature
mean_abs_shap = np.mean([np.abs(sv) for sv in shap_values], axis=(0, 1))

# Mostrar las 20 posiciones más influyentes
top20_idx = np.argsort(mean_abs_shap)[-20:][::-1]
plt.figure(figsize=(10, 5))
plt.bar(range(20), mean_abs_shap[top20_idx], color='steelblue')
plt.xlabel('Posición del coeficiente MFCC (índice en el vector de 25.000)')
plt.ylabel('Importancia media |SHAP|')
plt.title('Top 20 coeficientes MFCC más influyentes para la predicción')
plt.tight_layout()
plt.savefig('shap_importancia.png', dpi=150)
plt.show()
print('SHAP completado.')

---
# PARTE 5: SISTEMA DE PREDICCIÓN Y EXPORTACIÓN

In [ ]:
def predict_genre(song_path, model=model, genre_names=genre_names):
    """Predice el género de una canción y muestra las probabilidades."""
    feat = extract_features_song(song_path).reshape(1, -1)
    probs = model.predict(feat, verbose=0)[0]
    idx = np.argmax(probs)
    
    print(f'Canción:        {os.path.basename(song_path)}')
    print(f'Género predicho: {genre_names[idx].upper()} ({probs[idx]*100:.1f}%)')
    print('\nProbabilidades por género:')
    for g, p in sorted(zip(genre_names, probs), key=lambda x: -x[1]):
        bar = '█' * int(p * 40)
        print(f'  {g:12s}: {p*100:5.1f}% {bar}')
    return genre_names[idx]


# Ejemplo de uso
test_song = glob.glob(DATA_AUDIO + 'jazz/*.wav')[0]
predict_genre(test_song)

In [ ]:
# Guardar modelo y nombres de géneros
model.save('modelo_genero_musical.h5')
np.save('genre_names.npy', genre_names)
print('Modelo guardado: modelo_genero_musical.h5')
print('Géneros guardados: genre_names.npy')